# Final holdout and latest regional forecast

This notebook evaluates the frozen `horizon_specific_naive_baseline_v1` procedure once on the final holdout, audits its development-calibrated uncertainty intervals, and produces unscored forecasts from origin `2026Q2`. The procedure uses persistence at H1, H2, and H4 and annual seasonal naive at H3. It does not tune a model or interval after observing holdout results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path

DRIVE_PROJECT_ROOT = Path('/content/drive/MyDrive/ds_portfolio/project_02_coal_production_forecasting')
DATA_ROOT = DRIVE_PROJECT_ROOT / 'data'
RUNS_ROOT = DRIVE_PROJECT_ROOT / 'runs'

In [ ]:
import subprocess
import sys

REPO_URL = 'https://github.com/ahmaddshbg-blip/regional-coal-production-forecasting.git'
REPO_DIR = Path('/content/regional-coal-production-forecasting')
if (REPO_DIR / '.git').is_dir():
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-r', str(REPO_DIR / 'requirements-lock.txt')],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', 'scikit-learn==1.9.1'],
    check=True,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '-e', str(REPO_DIR), '--no-deps'],
    check=True,
)
source_root = str(REPO_DIR / 'src')
if source_root not in sys.path:
    sys.path.insert(0, source_root)
revision = subprocess.run(
    ['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'],
    check=True, capture_output=True, text=True,
).stdout.strip()
print('Code revision:', revision)

In [ ]:
import os

os.chdir(REPO_DIR)
os.environ['PROJECT_DATA_ROOT'] = str(DATA_ROOT)
os.environ['PROJECT_RUNS_ROOT'] = str(RUNS_ROOT)

import coal_forecasting

print('Python:', sys.version)
print('coal_forecasting:', coal_forecasting.__file__)
test_process = subprocess.run(
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
    cwd=REPO_DIR, text=True, capture_output=True,
)
print('TEST STDOUT')
print(test_process.stdout)
print('TEST STDERR')
print(test_process.stderr)
if test_process.returncode != 0:
    raise RuntimeError(
        f'Unit tests failed with exit code {test_process.returncode}. '
        'Use the printed test output to identify the first failure.'
    )

## One-time holdout boundary

The first successful run is irreversible: it writes a durable marker before reading holdout target magnitudes. Review `docs/final_baseline_procedure.md`, then change `OPEN_FINAL_HOLDOUT` to `True` for the authorized first run. A later rerun validates and reuses the passed artifacts; it does not rescore the holdout. A marker without a passed manifest blocks automatic retry and requires a documented recovery decision.


In [ ]:
from coal_forecasting import run_final_baseline_evaluation

OPEN_FINAL_HOLDOUT = False
result = run_final_baseline_evaluation(
    REPO_DIR / 'configs' / 'project.json',
    REPO_DIR / 'configs' / 'final_baseline.json',
    open_holdout=OPEN_FINAL_HOLDOUT,
    root=REPO_DIR,
)
print('Access mode:', result['access_mode'])
print('Run:', result['manifest']['run_id'])
print('Manifest:', result['manifest']['manifest_path'])

## Lineage and safety evidence


In [ ]:
import pandas as pd
from IPython.display import display

manifest = result['manifest']
lineage = pd.DataFrame.from_records([
    {
        'run_id': manifest['run_id'],
        'procedure_id': manifest['procedure_id'],
        'source_data_run': manifest['source_data_run'],
        'source_baseline_run': manifest['source_baseline_run'],
        'snapshot': manifest['checkpoint_id'],
        'holdout_opened': manifest['checks']['holdout_opened'],
        'candidate_confirmation_opened': manifest['checks']['candidate_confirmation_opened'],
        'latest_forecasts_unscored': manifest['checks']['latest_forecasts_unscored'],
        'git_commit': manifest['git']['commit'],
        'dirty_worktree': manifest['git']['dirty'],
    }
])
display(lineage)

## Final holdout point accuracy


In [ ]:
point_metrics = result['holdout_metrics'][
    [
        'horizon', 'forecast_rows', 'scored_cells', 'prediction_coverage',
        'target_availability', 'wape', 'median_state_mase',
        'aggregate_signed_bias', 'mae', 'rmse',
    ]
].copy()
display(point_metrics.style.format({
    'prediction_coverage': '{:.1%}',
    'target_availability': '{:.1%}',
    'wape': '{:.2%}',
    'median_state_mase': '{:.3f}',
    'aggregate_signed_bias': '{:.2%}',
    'mae': '{:,.0f}',
    'rmse': '{:,.0f}',
}))

## Interval calibration audit


In [ ]:
interval_metrics = result['holdout_interval_metrics'].copy()
pooled = interval_metrics.loc[interval_metrics['evaluation_scope'].eq('pooled')].copy()
pooled['guardrail_lower'] = pooled['nominal_coverage'].map({0.80: 0.75, 0.95: 0.90})
pooled['guardrail_upper'] = pooled['nominal_coverage'].map({0.80: 0.85, 0.95: 0.98})
pooled['within_predeclared_guardrail'] = pooled['empirical_coverage'].between(
    pooled['guardrail_lower'], pooled['guardrail_upper'], inclusive='both'
)
display(interval_metrics.style.format({
    'nominal_coverage': '{:.0%}',
    'empirical_coverage': '{:.2%}',
    'mean_width': '{:,.0f}',
    'median_width': '{:,.0f}',
    'mean_relative_width': '{:.2f}',
    'mean_winkler_score': '{:,.0f}',
}))
display(pooled[
    ['nominal_coverage', 'empirical_coverage', 'guardrail_lower',
     'guardrail_upper', 'within_predeclared_guardrail']
].style.format({
    'nominal_coverage': '{:.0%}',
    'empirical_coverage': '{:.2%}',
    'guardrail_lower': '{:.0%}',
    'guardrail_upper': '{:.0%}',
}))

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(point_metrics['horizon'], point_metrics['wape'], color='#2563eb')
axes[0].set_title('Final holdout WAPE by horizon')
axes[0].set_xlabel('Forecast horizon')
axes[0].set_ylabel('WAPE')
axes[0].set_xticks([1, 2, 3, 4])
axes[0].yaxis.set_major_formatter(PercentFormatter(1.0))
by_horizon = interval_metrics.loc[interval_metrics['evaluation_scope'].eq('horizon')]
for coverage, group in by_horizon.groupby('nominal_coverage'):
    axes[1].plot(
        group['horizon'], group['empirical_coverage'], marker='o',
        label=f'{coverage:.0%} interval',
    )
    axes[1].axhline(coverage, color='#6b7280', linewidth=0.8, linestyle='--')
axes[1].set_title('Holdout interval coverage')
axes[1].set_xlabel('Forecast horizon')
axes[1].set_ylabel('Empirical coverage')
axes[1].set_xticks([1, 2, 3, 4])
axes[1].yaxis.set_major_formatter(PercentFormatter(1.0))
axes[1].legend(frameon=False)
for axis in axes:
    axis.grid(axis='y', alpha=0.25)
fig.tight_layout()

## Stability across origins and states


In [ ]:
origin_metrics = result['holdout_origin_metrics'].copy()
state_metrics = result['holdout_state_metrics'].copy()
fig, ax = plt.subplots(figsize=(12, 4.5))
for horizon, group in origin_metrics.groupby('horizon'):
    ax.plot(group['origin_date'], group['wape'], marker='o', label=f'H{horizon}')
ax.set_title('Final holdout error by forecast origin')
ax.set_xlabel('Origin')
ax.set_ylabel('WAPE')
ax.yaxis.set_major_formatter(PercentFormatter(1.0))
ax.grid(axis='y', alpha=0.25)
ax.legend(frameon=False, ncol=4)
fig.tight_layout()
display(
    state_metrics.sort_values(['horizon', 'actual_volume'], ascending=[True, False])
    [['horizon', 'state_code', 'scored_cells', 'actual_volume', 'wape', 'mase', 'signed_mean_error']]
    .head(24)
    .style.format({
        'actual_volume': '{:,.0f}', 'wape': '{:.2%}', 'mase': '{:.3f}',
        'signed_mean_error': '{:,.0f}',
    })
)

## Latest-origin planning view

The ranking below uses only H1-H2 forecast volume. It identifies regions for review; it is not a forecast of equipment demand, staffing, service workload, revenue, or causal impact.


In [ ]:
latest = result['latest_forecasts'].copy()
planning = result['latest_planning_summary'].copy()
display(planning.head(15).style.format({
    'h1_forecast': '{:,.0f}', 'h1_lower_80': '{:,.0f}', 'h1_upper_80': '{:,.0f}',
    'h2_forecast': '{:,.0f}', 'h2_lower_80': '{:,.0f}', 'h2_upper_80': '{:,.0f}',
    'near_term_mean_forecast': '{:,.0f}', 'near_term_mean_width_80': '{:,.0f}',
}))
display(latest[
    ['source_snapshot', 'revision_status', 'origin', 'target_period', 'horizon',
     'state_code', 'source_model', 'forecast', 'lower_80', 'upper_80',
     'lower_95', 'upper_95', 'interval_scale_source']
].head(20))

In [ ]:
top_states = planning.head(12)['state_code'].tolist()
chart = latest.loc[
    latest['state_code'].isin(top_states) & latest['horizon'].isin([1, 2])
].copy()
positions = {state: index for index, state in enumerate(reversed(top_states))}
fig, ax = plt.subplots(figsize=(11, 6))
colors = {1: '#2563eb', 2: '#b45309'}
offsets = {1: -0.12, 2: 0.12}
for horizon, group in chart.groupby('horizon'):
    y = [positions[state] + offsets[int(horizon)] for state in group['state_code']]
    lower_error = group['forecast'] - group['lower_80']
    upper_error = group['upper_80'] - group['forecast']
    ax.errorbar(
        group['forecast'], y, xerr=[lower_error, upper_error], fmt='o',
        capsize=3, color=colors[int(horizon)], label=f'H{horizon} with 80% interval',
    )
ax.set_yticks(range(len(top_states)), labels=list(reversed(top_states)))
ax.set_title('Near-term state production forecasts for review')
ax.set_xlabel('Coal production, short tons')
ax.grid(axis='x', alpha=0.25)
ax.legend(frameon=False)
fig.tight_layout()

## Interpretation boundary

Holdout metrics estimate latest-vintage out-of-sample forecast performance for the fixed state-quarter production target. The latest forecasts are unscored planning signals from snapshot `20260920_1b3a8424_38776a23`; `2026Q2` may be revised by MSHA. H1-H2 receive operating attention, while H3-H4 remain visible because the one-to-four-quarter objective was frozen before results. Operational decisions still require mine plans, contracts, fleet, workforce, maintenance, commodity-market, and cost data that are outside this project.
